In [1]:
import os
import os.path as osp
import sys

sys.path.append("../src")

In [2]:
from config import CFG

In [3]:
import mlflow
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

/Users/dric/projects/research/LLMs/chat_app/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
mlflow.langchain.autolog()

In [5]:
model = init_chat_model(
    model = CFG.BASE_MODEL,
    model_provider=CFG.MODEL_PROVIDER,
    base_url=CFG.VLLM_BASE_URL,
    api_key=CFG.VLLM_API_KEY,
    temperature=CFG.GENERATION_TEMPERATURE
)

In [6]:
mlflow.set_experiment("chat-app")

2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/13 13:55:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/13 13:55:31 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/13 13:55:31 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/Users/dric/projects/research/LLMs/chat_app/notebooks/mlruns/1', creation_time=1770983786745, experiment_id='1', last_update_time=1770983786745, lifecycle_stage='active', name='chat-app', tags={}>

## Chat 

In [7]:
system_msg = SystemMessage(
    f"""
    You are a Sata Scientist with deep expertise in elections data and an SQL Expert.
    Based on a database with tables {CFG.ALLOWED_TABLES} among others, you directly answer questions about elections only.
    
    First, you classify the user's election query into one category:
        - AGGREGATION: Sums, counts, averages (e.g., 'total votes', 'turnout average').
        - RANKING: Comparisons, top/bottom lists (e.g., 'who won', 'best party').
        - CHART: Requests for distribution, trends, or visualizations.
        - GENERAL: Simple fact retrieval.
        - INVALID: Not about the election.
    
    Expected Output: Generate a single valid SQL SELECT statement. 
        - Always include a LIMIT {CFG.SQL_MAX_LIMIT}.
    """
)

human_msg = HumanMessage("Who won in Commune Cocody?")

messages = [system_msg, human_msg]

In [8]:
%%time

response = model.invoke(messages)
print(response.content)

CPU times: user 88.1 ms, sys: 56.5 ms, total: 145 ms
Wall time: 499 ms


NotFoundError: Error code: 404 - {'error': {'message': 'The model `Qwen/Qwen3-0.6B #Qwen/Qwen3-4B-Instruct-2507 facebook/opt-125m mistralai/Mistral-7B-Instruct-v0.3" #deepseek-ai/DeepSeek-R1-0528-Qwen3-8B #mistralai/Mistral-7B-Instruct-v0.3 openai/gpt-oss-20b #microsoft/Phi-4-mini-instruct` does not exist.', 'type': 'NotFoundError', 'param': 'model', 'code': 404}}

## Follow-up 

In [ ]:
messages.append(response)
messages.append(HumanMessage(
    "How many votes did candidate Ahmed Bamba get and what was the total number of votes?"
))

SyntaxError: invalid syntax (4062125449.py, line 1)

In [ ]:
%%time

response = model.invoke(messages)
print(response.content)